# Customer Churn Prediction — Step 5: Feature Engineering & ML Preprocessing

**Dataset:** `data/telco_churn_cleaned.csv` — produced by Step 3 (Data Cleaning)  
**Output:** `data/preprocessed_splits.pkl` — serialised train/test arrays and the unfitted `ColumnTransformer`  

### What this notebook does
1. Loads the cleaned dataset.
2. Defines the feature matrix `X` and target vector `y`.
3. Encodes the target (`No` → 0, `Yes` → 1).
4. Separates features into numerical and categorical groups.
5. Splits the data into train (80%) and test (20%) sets, **stratified** on `y`.
6. Constructs a `scikit-learn` `ColumnTransformer` pipeline (numerical: `StandardScaler`; categorical: `OneHotEncoder`). The pipeline is **defined** here but **not fitted** — fitting happens inside each model pipeline in Step 6 to prevent data leakage.
7. Verifies missing-value counts and reports all shapes and distributions.
8. Serialises the splits and the unfitted preprocessor for reproducible use in Step 6.

### What this notebook does NOT do
- Does **not** train any machine learning model.
- Does **not** evaluate model performance.
- Does **not** fit the `ColumnTransformer` to any data.
- Does **not** manually encode ordinal integers for nominal categorical variables.
- Does **not** modify the cleaned CSV.

---

## 1. Setup — Import Libraries and Load Cleaned Data

We import only the libraries needed for data handling and preprocessing definition.  
`sklearn.pipeline` and `sklearn.compose` are the building blocks of our preprocessing pipeline.  
`joblib` is used to serialise the splits and the unfitted preprocessor to disk so that Step 6 can load them without re-running this notebook.

In [2]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

RANDOM_STATE = 42

# ── Resolve paths ─────────────────────────────────────────────────────────────
NOTEBOOK_DIR  = os.path.abspath('')
PROJECT_ROOT  = os.path.dirname(NOTEBOOK_DIR) if os.path.basename(NOTEBOOK_DIR) == 'notebook' else NOTEBOOK_DIR
CLEAN_PATH    = os.path.join(PROJECT_ROOT, 'data', 'telco_churn_cleaned.csv')
SPLITS_PATH   = os.path.join(PROJECT_ROOT, 'data', 'preprocessed_splits.pkl')

print('Cleaned data path :', CLEAN_PATH)
print('Splits output path:', SPLITS_PATH)
print('File exists:', os.path.exists(CLEAN_PATH))

Cleaned data path : c:\Users\Subham\OneDrive\Documents\Desktop\Customer_Churn_Project\data\telco_churn_cleaned.csv
Splits output path: c:\Users\Subham\OneDrive\Documents\Desktop\Customer_Churn_Project\data\preprocessed_splits.pkl
File exists: True


In [3]:
# Load the cleaned dataset — never modified in this notebook
df = pd.read_csv(CLEAN_PATH)

print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Missing values in dataset: {df.isnull().sum().sum()}')

Loaded: 7,021 rows × 20 columns
Missing values in dataset: 0


---
## 2. Define X and y

**`X` (feature matrix)** contains all 19 predictor columns — every column except `Churn`.  
`customerID` was already removed in Step 3; it carries no predictive signal and would cause a unique-value explosion in any encoder.

**`y` (target vector)** is derived from the `Churn` column.  
We map the string values to integers:  
- `'No'`  → **0** (customer was retained)  
- `'Yes'` → **1** (customer churned)  

Using 0/1 integers is the standard expectation of scikit-learn classifiers and makes the target directly interpretable as a probability proxy in models like Logistic Regression.  
We verify the class distribution immediately after encoding to confirm the mapping is correct.

In [4]:
# Feature matrix — all columns except the target
X = df.drop(columns=['Churn'])

# Target vector — binary encoded
y = (df['Churn'] == 'Yes').astype(int)

print(f'X shape: {X.shape}   (rows × features)')
print(f'y shape: {y.shape}')
print()
print('y value counts (0 = No Churn, 1 = Churned):')
counts = y.value_counts().sort_index()
pcts   = y.value_counts(normalize=True).sort_index().mul(100).round(2)
summary = pd.DataFrame({'Count': counts, 'Percentage (%)': pcts})
display(summary)

X shape: (7021, 19)   (rows × features)
y shape: (7021,)

y value counts (0 = No Churn, 1 = Churned):


,Count,Percentage (%)
Churn,,
0,5164,73.55
1,1857,26.45


**Verification:** The encoded distribution matches the Step 4 EDA result exactly:  
- **0 (No Churn):** ~73.6% of all customers  
- **1 (Churned):** ~26.4% of all customers  

The encoding is correct.

---
## 3. Separate Numerical and Categorical Features

Scikit-learn's `ColumnTransformer` requires us to specify which columns get which transformer.  
We separate the 19 predictor columns into two lists:

**Numerical features** (`float64` / `int64`):  
- `SeniorCitizen` — kept as a binary integer (0/1) rather than a one-hot encoded categorical, as decided in Step 3. `StandardScaler` will not distort its meaning since it only has two values.
- `tenure` — continuous integer (0–72 months)  
- `MonthlyCharges` — continuous float  
- `TotalCharges` — continuous float  

**Categorical features** (all remaining `object` dtype columns):  
15 columns — all nominal (no meaningful ordinal ordering), so `OneHotEncoder` is the correct choice.  

We do **not** manually assign integer codes such as `Month-to-month=0, One year=1, Two year=2` for `Contract`.  
That would impose a false ordinal relationship and scale on a nominally encoded variable, which would distort distance-based models and coefficient interpretation.

In [5]:
# Numerical features: 4 columns (SeniorCitizen treated as numeric binary)
numerical_features = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

# Categorical features: all remaining predictor columns
categorical_features = [col for col in X.columns if col not in numerical_features]

print(f'Numerical features  ({len(numerical_features)}): {numerical_features}')
print()
print(f'Categorical features ({len(categorical_features)}):')
for col in categorical_features:
    n_unique = X[col].nunique()
    vals = sorted(X[col].unique().tolist())
    print(f'  {col:<22} {n_unique} unique values: {vals}')

Numerical features  (4): ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features (15):
  gender                 2 unique values: ['Female', 'Male']
  Partner                2 unique values: ['No', 'Yes']
  Dependents             2 unique values: ['No', 'Yes']
  PhoneService           2 unique values: ['No', 'Yes']
  MultipleLines          3 unique values: ['No', 'No phone service', 'Yes']
  InternetService        3 unique values: ['DSL', 'Fiber optic', 'No']
  OnlineSecurity         3 unique values: ['No', 'No internet service', 'Yes']
  OnlineBackup           3 unique values: ['No', 'No internet service', 'Yes']
  DeviceProtection       3 unique values: ['No', 'No internet service', 'Yes']
  TechSupport            3 unique values: ['No', 'No internet service', 'Yes']
  StreamingTV            3 unique values: ['No', 'No internet service', 'Yes']
  StreamingMovies        3 unique values: ['No', 'No internet service', 'Yes']
  Contract               3 unique v

---
## 4. Train / Test Split

We split the data **before** any preprocessing is fitted.  
This is the foundational safeguard against **data leakage** — if a scaler or encoder were fitted on the full dataset first, the test set statistics would silently influence the transformation, giving an optimistically biased view of model performance.

**Split parameters:**
- `test_size = 0.20` — 20% test, 80% train. With 7,021 rows this gives ~5,616 training samples and ~1,405 test samples — enough for stable model training and reliable evaluation.
- `random_state = 42` — ensures the split is fully reproducible across runs and across machines.
- `stratify = y` — **stratification preserves the class ratio** in both the train and test sets. Without stratification, a random split could by chance assign a disproportionate share of the minority class (churned customers, 26.4%) to one partition. Stratification guarantees both partitions reflect the 73.6 / 26.4 split seen in the full dataset, making train and test comparable.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test  shape: {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test  shape: {y_test.shape}')

X_train shape: (5616, 19)
X_test  shape: (1405, 19)
y_train shape: (5616,)
y_test  shape: (1405,)


In [7]:
# Verify that stratification preserved the class ratio in both partitions
def class_distribution_report(series, name):
    counts = series.value_counts().sort_index()
    pcts   = series.value_counts(normalize=True).sort_index().mul(100).round(2)
    df_out = pd.DataFrame({'Count': counts, 'Percentage (%)': pcts})
    df_out.index.name = 'Churn'
    print(f'--- {name} ---')
    display(df_out)

class_distribution_report(y_train, 'y_train')
class_distribution_report(y_test,  'y_test')

--- y_train ---


,Count,Percentage (%)
Churn,,
0,4131,73.56
1,1485,26.44


--- y_test ---


,Count,Percentage (%)
Churn,,
0,1033,73.52
1,372,26.48


**Verification:** Both `y_train` and `y_test` reflect the same ~73.6% / 26.4% split as the full dataset — stratification is working correctly.

---
## 5. Check Missing Values in Splits

The cleaned dataset from Step 3 contains no missing values. However, we verify this explicitly on `X_train` and `X_test` before building the preprocessing pipeline.  
This check also documents that the `SimpleImputer` steps inside the pipeline are **defensive** — they guard against any future upstream change that could introduce nulls — rather than performing active imputation on this dataset.

In [8]:
print('=== Missing values in X_train ===')
missing_train = X_train.isnull().sum()
missing_train_nonzero = missing_train[missing_train > 0]
if missing_train_nonzero.empty:
    print('No missing values detected in X_train.')
else:
    display(missing_train_nonzero)

print()
print('=== Missing values in X_test ===')
missing_test = X_test.isnull().sum()
missing_test_nonzero = missing_test[missing_test > 0]
if missing_test_nonzero.empty:
    print('No missing values detected in X_test.')
else:
    display(missing_test_nonzero)

=== Missing values in X_train ===
No missing values detected in X_train.

=== Missing values in X_test ===
No missing values detected in X_test.


---
## 6. Build the Preprocessing Pipeline

We build a `ColumnTransformer` that contains two sub-pipelines — one for numerical features and one for categorical features.  

### Numerical pipeline
1. **`SimpleImputer(strategy='median')`** — defensive step; fills any future nulls with the column median (robust to outliers).  
2. **`StandardScaler()`** — subtracts the column mean and divides by standard deviation, producing zero-mean unit-variance features. Required for distance-sensitive models (Logistic Regression, SVM, KNN). Tree-based models (Random Forest, XGBoost) are invariant to scaling, but standardisation does not hurt them, so a single consistent pipeline works across all model types.

### Categorical pipeline
1. **`SimpleImputer(strategy='most_frequent')`** — defensive step; fills any future nulls with the most common category.  
2. **`OneHotEncoder(handle_unknown='ignore', sparse_output=False)`** — creates a binary indicator column for each unique category value.  
   - `handle_unknown='ignore'` ensures the encoder silently outputs an all-zeros row for any category value encountered in test or production that was not seen during training (instead of raising an error).  
   - `sparse_output=False` returns a dense NumPy array, which is compatible with all downstream estimators.
   - We do **not** use `drop='first'` here. Dropping the first category is commonly done in linear models to avoid perfect multicollinearity; however, tree-based models work better with all categories present, and regularised linear models (LogisticRegression with `C`) handle the multicollinearity automatically. Keeping all columns makes the pipeline compatible with both.

### Why the pipeline is NOT fitted here
The `ColumnTransformer` is only **defined** — not fitted — in this notebook.  
In Step 6, each model is wrapped as:  
```python
Pipeline(steps=[('preprocessor', preprocessor), ('model', SomeClassifier())])
```
The full pipeline is then fitted **only on `X_train`** via `.fit(X_train, y_train)`.  
This means the scaler's mean/std and the encoder's category vocabulary are learned solely from training data, and the test set transformation uses those training-derived parameters — which is the correct and leakage-free approach.

In [9]:
# ── Numerical sub-pipeline ────────────────────────────────────────────────────
numerical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# ── Categorical sub-pipeline ──────────────────────────────────────────────────
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# ── ColumnTransformer — combines both pipelines ───────────────────────────────
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline,  numerical_features),
        ('cat', categorical_pipeline, categorical_features),
    ],
    remainder='drop'       # any column not listed above is dropped
)

print('ColumnTransformer defined (NOT yet fitted):')
print(preprocessor)

ColumnTransformer defined (NOT yet fitted):
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['SeniorCitizen', 'tenure', 'MonthlyCharges',
                                  'TotalCharges']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['gender', 'Partner', 'Dependents',
                                  'PhoneService', 'Multip

---
## 7. Preview: Expected Output Shape After Preprocessing

Before Step 6, it is helpful to confirm the expected number of output columns so there are no surprises when models are trained.  

We can compute this exactly by temporarily fitting the preprocessor on `X_train` (just for inspection), checking the output shape, then discarding the fitted object — we do not use this fitted version for modelling.

**Expected columns:**  
- 4 numerical columns (scaled, passed through 1:1)  
- 41 one-hot encoded columns from 15 categorical features (2 binary features × 2 cols each + 9 three-value features × 3 cols each + 1 four-value feature × 4 cols)  
- **Total: 45 output columns**

In [10]:
# Fit a temporary copy ONLY for shape inspection — NOT the preprocessor used in Step 6
import copy
preprocessor_temp = copy.deepcopy(preprocessor)
X_train_transformed_preview = preprocessor_temp.fit_transform(X_train)

print(f'X_train after preprocessing: {X_train_transformed_preview.shape}')
print(f'  — {len(numerical_features)} numerical columns (scaled)')

# Retrieve the OHE feature names
ohe = preprocessor_temp.named_transformers_['cat'].named_steps['encoder']
ohe_feature_names = ohe.get_feature_names_out(categorical_features)
print(f'  — {len(ohe_feature_names)} one-hot encoded columns from {len(categorical_features)} categorical features')
print(f'  — Total output columns: {X_train_transformed_preview.shape[1]}')
print()
print('OHE column names:')
for name in ohe_feature_names:
    print(f'  {name}')

del preprocessor_temp, X_train_transformed_preview  # discard — not used for modelling

X_train after preprocessing: (5616, 45)
  — 4 numerical columns (scaled)
  — 41 one-hot encoded columns from 15 categorical features
  — Total output columns: 45

OHE column names:
  gender_Female
  gender_Male
  Partner_No
  Partner_Yes
  Dependents_No
  Dependents_Yes
  PhoneService_No
  PhoneService_Yes
  MultipleLines_No
  MultipleLines_No phone service
  MultipleLines_Yes
  InternetService_DSL
  InternetService_Fiber optic
  InternetService_No
  OnlineSecurity_No
  OnlineSecurity_No internet service
  OnlineSecurity_Yes
  OnlineBackup_No
  OnlineBackup_No internet service
  OnlineBackup_Yes
  DeviceProtection_No
  DeviceProtection_No internet service
  DeviceProtection_Yes
  TechSupport_No
  TechSupport_No internet service
  TechSupport_Yes
  StreamingTV_No
  StreamingTV_No internet service
  StreamingTV_Yes
  StreamingMovies_No
  StreamingMovies_No internet service
  StreamingMovies_Yes
  Contract_Month-to-month
  Contract_One year
  Contract_Two year
  PaperlessBilling_No
  Pape

**Confirmation:** The `ColumnTransformer` produces 45 output columns — 4 scaled numerical + 41 one-hot encoded — consistent with the manual count above.  
The unfitted `preprocessor` object is unchanged and ready for use in Step 6.

---
## 8. Full Summary Report

A consolidated report of all shapes, feature lists, and class distributions to serve as a reference when building models in Step 6.

In [11]:
print('=' * 60)
print('STEP 5 — PREPROCESSING SUMMARY')
print('=' * 60)
print()
print(f'Full dataset shape        : {df.shape}')
print(f'X shape                   : {X.shape}')
print(f'y shape                   : {y.shape}')
print()
print(f'Train set (X_train, y_train): {X_train.shape}, {y_train.shape}')
print(f'Test  set (X_test,  y_test) : {X_test.shape},  {y_test.shape}')
print()
print(f'Numerical features  ({len(numerical_features)})  : {numerical_features}')
print(f'Categorical features ({len(categorical_features)}): {categorical_features}')
print()
print('y_train class distribution:')
for val, cnt in y_train.value_counts().sort_index().items():
    pct = cnt / len(y_train) * 100
    label = 'No Churn' if val == 0 else 'Churned '
    print(f'  {val} ({label}): {cnt:,}  ({pct:.2f}%)')
print()
print('y_test class distribution:')
for val, cnt in y_test.value_counts().sort_index().items():
    pct = cnt / len(y_test) * 100
    label = 'No Churn' if val == 0 else 'Churned '
    print(f'  {val} ({label}): {cnt:,}  ({pct:.2f}%)')
print()
print(f'Preprocessor output cols  : 45 (4 numerical + 41 OHE)')
print(f'Random state              : {RANDOM_STATE}')
print(f'Test fraction             : 20%')
print(f'Stratified split          : Yes')
print(f'Preprocessor fitted       : No — will be fitted in Step 6 per model')
print('=' * 60)

STEP 5 — PREPROCESSING SUMMARY

Full dataset shape        : (7021, 20)
X shape                   : (7021, 19)
y shape                   : (7021,)

Train set (X_train, y_train): (5616, 19), (5616,)
Test  set (X_test,  y_test) : (1405, 19),  (1405,)

Numerical features  (4)  : ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features (15): ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

y_train class distribution:
  0 (No Churn): 4,131  (73.56%)
  1 (Churned ): 1,485  (26.44%)

y_test class distribution:
  0 (No Churn): 1,033  (73.52%)
  1 (Churned ): 372  (26.48%)

Preprocessor output cols  : 45 (4 numerical + 41 OHE)
Random state              : 42
Test fraction             : 20%
Stratified split          : Yes
Preprocessor fitted       : No — will be fitted in Step 6 per 

---
## 9. Save Artifacts for Step 6

We serialise the raw (unencoded) `X_train`, `X_test`, `y_train`, `y_test` and the **unfitted** `preprocessor` to a single `pkl` file using `joblib`.  

Step 6 loads this file instead of re-running the split, which guarantees that every model is evaluated on exactly the same train and test rows — a critical requirement for fair comparison.

We save `X_train`/`X_test` as raw DataFrames (not yet transformed) because each model's `Pipeline` in Step 6 will call `.fit_transform` on `X_train` internally, learning the scaling parameters and encoder vocabulary from training data only.

In [12]:
artifacts = {
    'X_train':       X_train,
    'X_test':        X_test,
    'y_train':       y_train,
    'y_test':        y_test,
    'preprocessor':  preprocessor,          # unfitted ColumnTransformer
    'numerical_features':   numerical_features,
    'categorical_features': categorical_features,
    'random_state':  RANDOM_STATE,
}

joblib.dump(artifacts, SPLITS_PATH)
print(f'Artifacts saved to: {SPLITS_PATH}')

# Reload and verify
loaded = joblib.load(SPLITS_PATH)
print()
print('Verification — keys in saved file:', list(loaded.keys()))
print(f'  X_train: {loaded["X_train"].shape}')
print(f'  X_test : {loaded["X_test"].shape}')
print(f'  y_train: {loaded["y_train"].shape}')
print(f'  y_test : {loaded["y_test"].shape}')
print(f'  preprocessor fitted: {hasattr(loaded["preprocessor"], "transformers_")}')

Artifacts saved to: c:\Users\Subham\OneDrive\Documents\Desktop\Customer_Churn_Project\data\preprocessed_splits.pkl

Verification — keys in saved file: ['X_train', 'X_test', 'y_train', 'y_test', 'preprocessor', 'numerical_features', 'categorical_features', 'random_state']
  X_train: (5616, 19)
  X_test : (1405, 19)
  y_train: (5616,)
  y_test : (1405,)
  preprocessor fitted: False


---
## 10. Feature Engineering and Preprocessing Decisions

This section documents every preprocessing decision made in this notebook, the reasoning behind it, and how it prevents data leakage.

---

### Why `customerID` is not used
`customerID` is an arbitrary identifier assigned to each customer record. It carries no information about a customer's behaviour, usage patterns, or service configuration — it is purely an administrative key.  
Including it in a model would either (a) be ignored by most algorithms as noise, or (b) cause a tree-based model to memorise training rows by ID, severely overfitting. It was removed in Step 3 and is absent from the cleaned dataset.

---

### Why `Churn` is encoded as 0 / 1
Scikit-learn classifiers expect the target variable to be a numeric type. The string values `'No'` and `'Yes'` would raise errors in most estimators.  
The mapping `'No'→0, 'Yes'→1` is the universally understood binary convention:  
- **0** = negative class (no event occurred — customer was retained)  
- **1** = positive class (event occurred — customer churned)  
Keeping it as integer rather than a 2-column OHE representation avoids unnecessary complexity and is directly interpretable as a predicted probability when using `predict_proba`.

---

### Why categorical variables use OneHotEncoder (not integer mapping)
All 15 categorical predictor columns are **nominal** — their categories have no inherent numerical order.  
Assigning arbitrary integers (e.g., `Contract: Month-to-month=0, One year=1, Two year=2`) would impose an artificial ordinal relationship and a false magnitude difference. A logistic regression would incorrectly treat the integer-encoded value as a continuous score, and the distance between `One year` and `Two year` would be treated as equal to the distance between `Month-to-month` and `One year` — which has no real-world justification.  
One-hot encoding converts each category to a separate binary column, allowing every model type to learn independent coefficients or splits for each category without any implied ordering.

---

### Why numerical variables are standardised
`tenure` (0–72), `MonthlyCharges` (18–119), `TotalCharges` (0–8,685) and `SeniorCitizen` (0/1) operate on very different scales.  
Models that compute distances or rely on gradient magnitude — Logistic Regression, Support Vector Machines, K-Nearest Neighbours — are sensitive to feature scale. Without standardisation, `TotalCharges` (up to 8,685) would numerically dominate `SeniorCitizen` (0 or 1), causing the model to effectively ignore low-magnitude features.  
`StandardScaler` centres each feature at zero (subtracts mean) and scales it to unit variance (divides by std), putting all features on a comparable footing.  
Tree-based models (Random Forest, XGBoost) are invariant to monotone transformations of individual features, so standardisation does not hurt them — a single preprocessing pipeline works for all model types.

---

### Why the train/test split happens before preprocessing is fitted — and how this prevents data leakage
**Data leakage** occurs when information from the test set flows into the model training process, leading to overly optimistic evaluation metrics that do not reflect real-world performance.

If we were to fit `StandardScaler` on the full dataset (`X`) before splitting:  
- The scaler's mean and standard deviation would incorporate test-set values.  
- When the model is evaluated on the test set, the test data would have already "influenced" its own normalisation parameters.  
- The model would appear to perform better than it actually would on unseen production data.

The correct order is:
1. **Split** → produces `X_train` and `X_test` with no knowledge of each other.
2. **Fit** the preprocessor **on `X_train` only** — mean, std, and encoder vocabulary are derived entirely from training rows.
3. **Transform** both `X_train` and `X_test` using those training-derived parameters — the test set is normalised using training statistics, exactly as a deployed model would normalise unseen future data.

Using scikit-learn's `Pipeline` enforces this automatically: calling `pipeline.fit(X_train, y_train)` fits the preprocessor on `X_train`, and calling `pipeline.predict(X_test)` applies the already-fitted preprocessing to `X_test` without re-fitting.

---

### Why stratification is used
The dataset is moderately imbalanced: ~73.6% No Churn, ~26.4% Churned.  
A purely random split has a small but real chance of producing a test set with a disproportionate share of the minority class (churned customers).  
For example, in a random split the test set might contain 30% churned customers, making the evaluation metrics on the test set incomparable to metrics computed on the training set or on future production data.  
`stratify=y` ensures that both `y_train` and `y_test` preserve the same 73.6 / 26.4 ratio as the original dataset — the split is representative and the evaluation will not be accidentally inflated or deflated by a class-ratio artefact.